In [1]:
print("Google Gen AI notebook loaded")

Google Gen AI notebook loaded


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [14]:
import json
import os
import random
import time
import estnltk
import tqdm

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [4]:
"""
Google Gen AI setup for Estonian marked-word rewriting.

This cell configures the Gen AI client, reads the API key from the environment,
and prepares the model call for JSON output.
"""

load_dotenv(".env", verbose=True)

api_key = os.getenv("GOOGLE_API_KEY")
model = "gemini-3.1-flash-lite-preview"
client = genai.Client(api_key=api_key)

print(f"Using model: {model}")

Using model: gemini-3.1-flash-lite-preview


## Google Gen AI Prompting for Estonian Sentence Rewriting

This notebook uses Google's Gen AI SDK to rewrite Estonian sentences by replacing exactly one marked token in angle brackets (`<...>`) with a context-appropriate alternative.

The prompt asks the model to produce a JSON object with the rewritten sentence, a chosen replacement, and a short candidate list to make downstream parsing reliable.


### Testing on a single sentence


In [ ]:
# System prompt: enforce Estonian rewriting behaviour, case conditioning, and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.
Your task is to replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Keep the rewritten sentence natural and grammatically correct in Estonian.
- Preserve tense, number, case, agreement, punctuation, and capitalisation.
- The replacement must be a valid Estonian word form in one of these allowed cases only: nominative, genitive, partitive, illative.
- Do not generate replacements in any other morphological case.
- If the marked token is a proper name (person, place, organisation), replace it with another plausible proper name while still respecting the allowed-case constraint.
- Internally consider multiple alternatives, then choose the best one for the full sentence context. The chosen alternative must not be the same as the original marked word.
- Order the candidate alternatives by their suitability for the sentence context, with the chosen one as being first in the list of candidates.
- Return strict JSON only. Do not add explanations or markdown.

JSON schema:
{
  "id": string,
  "candidates": array of 10 strings,
  "chosen": string,
  "rewritten": string
}
"""

# Few-shot example to anchor the output format
few_shot_user = """Sentence ID: 0
Sentence: Ma näen <maja>.
"""

few_shot_assistant = json.dumps(
    {
        "id": "0",
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "häärberit",
        ],
        "chosen": "ehitist",
        "rewritten": "Ma näen ehitist.",
    },
    ensure_ascii=False,
    indent=2,
)

# Real input sentence
user_prompt = """Sentence ID: 1
Sentence: Samas on kõik uue kodu lähistel asuvad koolid sellised, mis võtavad <gümnaasiumi> vastu katsetega.
"""

contents = [
    types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
    types.Content(role="model", parts=[types.Part.from_text(text=few_shot_assistant)]),
    types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
]

In [20]:
# Request a JSON response from Gemini / Google Gen AI
response = client.models.generate_content(
    model=model,
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        # temperature=0.3,
        top_p=0.95,
        # top_k=40,
        max_output_tokens=100000,
        response_mime_type="application/json",
    ),
)

INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"


In [22]:
content = response.text
result = json.loads(content)

print("Raw JSON response:")
print(json.dumps(result, ensure_ascii=False, indent=2))

Raw JSON response:
{
  "id": "91",
  "candidates": [
    "sündmuskohal",
    "plahvatuses",
    "rünnakus",
    "õnnetuses",
    "kokkupõrkes",
    "intsidendis",
    "avarii",
    "krahhis",
    "pommiplahvatuses",
    "tulevahetuses"
  ]
}


In [24]:
result["candidates"]

['sündmuskohal',
 'plahvatuses',
 'rünnakus',
 'õnnetuses',
 'kokkupõrkes',
 'intsidendis',
 'avarii',
 'krahhis',
 'pommiplahvatuses',
 'tulevahetuses']

### Testing on a sample of sentences with batch processing and error handling


In [5]:
# Load the homonyms dataset to get the sentences we want to rewrite with Gen AI
homonyms_df = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_sentences.parquet"
)
display(homonyms_df.head())

,num,inflection_type,sentence,word,word_span,label,source
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [ ]:
# System prompt: enforce Estonian rewriting behaviour, case conditioning, and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.
Your task is to replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Keep the rewritten sentence natural and grammatically correct in Estonian.
- Preserve tense, number, case, agreement, punctuation, and capitalisation.
- The replacement must be a valid Estonian word form in one of these allowed cases only: nominative, genitive, partitive, illative.
- Do not generate replacements in any other morphological case.
- If the marked token is a proper name (person, place, organisation), replace it with another plausible proper name while still respecting the allowed-case constraint.
- Internally consider multiple alternatives, then choose the best one for the full sentence context. The chosen alternative must not be the same as the original marked word.
- Order the candidate alternatives by their suitability for the sentence context, with the chosen one as being first in the list of candidates.
- Return strict JSON only. Do not add explanations or markdown.

JSON schema:
{
  "id": string,
  "candidates": array of 10 strings,
  "chosen": string,
  "rewritten": string
}
"""

# Few-shot example to anchor the output format
few_shot_user = """Sentence ID: 0
Sentence: Ma näen <maja>.
"""

few_shot_assistant = json.dumps(
    {
        "id": "0",
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "häärberit",
        ],
        "chosen": "ehitist",
        "rewritten": "Ma näen ehitist.",
    },
    ensure_ascii=False,
    indent=2,
)

# Real input sentence
user_prompt = """Sentence ID: 1
Sentence: Samas on kõik uue kodu lähistel asuvad koolid sellised, mis võtavad <gümnaasiumi> vastu katsetega.
"""

contents = [
    types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
    types.Content(role="model", parts=[types.Part.from_text(text=few_shot_assistant)]),
    types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
]


# Create a user prompt template for the real sentences, using the same format as the few-shot example
def create_user_prompt(sentence_id, sentence, word_span):
    # Mark the target word with angle brackets
    start, end = word_span
    marked_sentence = (
        sentence[:start] + "<" + sentence[start:end] + ">" + sentence[end:]
    )
    return f"""Sentence ID: {sentence_id}
Sentence: {marked_sentence}
"""


def append_result_to_json(file_path, record):
    """Append one record to a JSON array stored in file_path."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    # If the file exists and is non-empty, load the existing data; otherwise start with an empty list
    if file_path.exists() and file_path.stat().st_size > 0:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = []
    else:
        data = []

    if not isinstance(data, list):
        data = []

    data.append(record)
    # Write the updated list back to the file with pretty formatting
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def rewrite_sentence_with_genai(sentence_id, sentence, word_span, metadata):
    """Rewrite one sentence by replacing the marked target word via Gemini."""
    user_prompt = create_user_prompt(
        sentence_id=sentence_id, sentence=sentence, word_span=word_span
    )
    # The few-shot example is included in every call to ensure the model understands the expected JSON format and the task.
    local_contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
        types.Content(
            role="model", parts=[types.Part.from_text(text=few_shot_assistant)]
        ),
        types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
    ]
    # Request a JSON response from Gemini / Google Gen AI
    response = client.models.generate_content(
        model=model,
        contents=local_contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            # temperature=0.3,
            top_p=0.92,
            max_output_tokens=100000,
            response_mime_type="application/json",
        ),
    )
    # Parse the JSON response and enrich it with metadata for later analysis
    content = response.text or "{}"
    parsed = json.loads(content)

    # Calculate the new word span in the rewritten sentence, if possible
    rewritten = parsed.get("rewritten", "")
    chosen = parsed.get("chosen", "")
    if rewritten and chosen:
        new_start = rewritten.find(chosen)
        if new_start != -1:
            new_end = new_start + len(chosen)
            parsed["new_word_span"] = [new_start, new_end]
        else:
            parsed["new_word_span"] = None
    else:
        parsed["new_word_span"] = None

    # Keep useful metadata for later analysis/debugging.
    parsed["source_sentence"] = sentence
    parsed["target_word"] = metadata.get("target_word", "")
    parsed["word_span"] = word_span.astype(
        int
    ).tolist()  # Convert numpy array to list for JSON serialization
    parsed["label"] = metadata.get("label", [])

    return parsed


def get_latest_processed_id(output_file):
    """Get the highest sentence ID that has already been processed and saved in the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        with open(output_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list) and len(data) > 0:
                return max(int(record["id"]) for record in data if "id" in record)
    return -1  # No valid records found, start from the beginning


def is_transient_error(exc: Exception) -> bool:
    """Heuristically detect retryable errors such as rate limit and network hiccups."""
    message = str(exc).lower()
    transient_markers = [
        "429",
        "rate",
        "quota",
        "too many requests",
        "timeout",
        "timed out",
        "connection",
        "temporar",
        "unavailable",
        "503",
        "502",
        "504",
        "internal",
    ]
    return any(marker in message for marker in transient_markers)


def rewrite_with_retry(sentence_id, sentence, word_span, metadata, config):
    """Call rewrite function with exponential backoff on transient errors."""
    for attempt in range(1, config["max_attempts"] + 1):
        try:
            return rewrite_sentence_with_genai(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                metadata=metadata,
            )
        except Exception as exc:
            should_retry = is_transient_error(exc) and attempt < config["max_attempts"]
            if not should_retry:
                raise

            backoff = min(
                config["max_backoff_seconds"],
                config["initial_backoff_seconds"] * (2 ** (attempt - 1)),
            )
            sleep_s = backoff + random.uniform(0.0, config["jitter_seconds"])
            print(
                f"Retry {attempt}/{config['max_attempts'] - 1} for sentence_id={sentence_id} "
                f"after transient error: {exc}. Sleeping {sleep_s:.2f}s..."
            )
            time.sleep(sleep_s)

In [7]:
example_sentence = homonyms_df.iloc[0]["sentence"]
example_span = homonyms_df.iloc[0]["word_span"]
example_id = 0
example_prompt = create_user_prompt(example_id, example_sentence, example_span)
print("Example user prompt for a real sentence:")
print(example_prompt)

Example user prompt for a real sentence:
Sentence ID: 0
Sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <võitja> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.



In [8]:
display(homonyms_df.head())

,num,inflection_type,sentence,word,word_span,label,source
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [9]:
# Create a small sample of sentences from homonym_df for testing the batch processing and error handling
# Extract n sentences from each inflection type (1, 16, 17, 19) to have a balanced test set
sample_df = (
    homonyms_df.groupby("inflection_type")
    .apply(lambda group: group.sample(n=25, random_state=seed))
    .reset_index(drop=True)
)
# Display inflection type distribution in the sample to verify balance
print("Inflection type distribution in the sample:")
print(sample_df["inflection_type"].value_counts())

Inflection type distribution in the sample:
inflection_type
1     25
16    25
17    25
19    25
Name: count, dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_30392\962437817.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.sample(n=25, random_state=seed))


In [ ]:
max_rows = 100  # Change or remove this limit for larger runs
output_file = (
    OUTPUT_DIR / f"genai_rewrites_sample_{max_rows}.json"
)  # Output file for results

# gemini-3.1-flash-lite-preview has strict request limits, so we throttle and retry safely.
config = {
    # Pacing configuration to avoid hitting rate limits
    "base_delay_seconds": 4.0,
    "jitter_seconds": 1.0,
    # Retry configuration for transient failures
    "max_attempts": 5,
    "initial_backoff_seconds": 2.0,
    "max_backoff_seconds": 60.0,
}

# Start from the next unprocessed sentence based on the output file contents
latest_id = get_latest_processed_id(output_file)
unprocessed_df = sample_df[sample_df.index > latest_id]
print(
    f"Starting processing from sentence ID > {latest_id + 1}. Total unprocessed sentences: {len(unprocessed_df)}"
)
if len(unprocessed_df) == 0:
    print("No unprocessed sentences found. All done!")
else:
    print("Next sentence to process:")
    display(unprocessed_df.head(1))

Starting processing from sentence ID > 0. Total unprocessed sentences: 100
Next sentence to process:


,num,inflection_type,sentence,word,word_span,label,source
0,1,1,"Foto : Priit Simson Euroopa Komisjon tahab, et rohkem tossavad autod oleks kõrgemini maksustatud.",Euroopa,"[20, 27]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [ ]:
# Batch rewrite loop: call Gen AI and append each result to a JSON file
processed = 0
for sentence_id, (sentence, word_span, word, label) in enumerate(
    zip(
        unprocessed_df["sentence"],
        unprocessed_df["word_span"],
        unprocessed_df["word"],
        unprocessed_df["label"],
    ),
    start=latest_id + 1,
):
    if sentence_id >= max_rows:
        break

    try:
        metadata = {
            "target_word": word,
            "label": label.tolist(),  # Convert numpy array to list for JSON serialization
        }
        result = rewrite_with_retry(
            sentence_id=sentence_id,
            sentence=sentence,
            word_span=word_span,
            config=config,
            metadata=metadata,
        )
        append_result_to_json(output_file, result)
        print(f"[{processed + 1}] Saved sentence_id={sentence_id}")
    except Exception as exc:
        error_record = {
            "id": str(sentence_id),
            "source_sentence": sentence,
            "target_word": word,
            "word_span": word_span.astype(int).tolist(),
            "error": str(exc),
        }
        append_result_to_json(output_file, error_record)
        print(f"[{processed + 1}] Error on sentence_id={sentence_id}: {exc}")

    processed += 1

    # Additional pacing between successful/failed items to avoid bursty traffic.
    if sentence_id + 1 < max_rows:
        sleep_s = config["base_delay_seconds"] + random.uniform(
            0.0, config["jitter_seconds"]
        )
        print(f"Sleeping {sleep_s:.2f}s before next request...")
        time.sleep(sleep_s)

print(f"Finished. Processed {processed} rows. Results appended to: {output_file}")

INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[1] Error on sentence_id=0: Extra data: line 26 column 1 (char 364)
Sleeping 4.24s before next request...
INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[2] Saved sentence_id=1
Sleeping 4.07s before next request...
INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[3] Error on sentence_id=2: Extra data: line 26 column 1 (char 446)
Sleeping 4.56s before next request...


KeyboardInterrupt: 

In [11]:
# Open the output file
if output_file.exists():
    with open(output_file, "r", encoding="utf-8") as f:
        data = json.load(f)
        # Iterate over the homonym dataset and add the form label and new word span as metadata to the corresponding result record
        for id, record in enumerate(data):
            matching_row = homonyms_df[homonyms_df.index == id]
            if not matching_row.empty:
                form_label = matching_row.iloc[0]["label"]
                record["form_label"] = form_label.tolist()
                record["id"] = str(id)  # Ensure the ID is a string for consistency
                # Calculate the new word span in the rewritten sentence, if possible
                rewritten = record.get("rewritten", "")
                chosen = record.get("chosen", "")
                if rewritten and chosen:
                    new_start = rewritten.find(chosen)
                    if new_start != -1:
                        new_end = new_start + len(chosen)
                        record["new_word_span"] = [new_start, new_end]
                    else:
                        record["new_word_span"] = None
                else:
                    record["new_word_span"] = None
            else:
                print(f"No matching row found in homonyms_df for record ID: {id}")
    # Save the enriched results back to the file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
# Open LLM sample annotation data
llm_samples_df = pd.read_json(output_file, orient="records")
cols = llm_samples_df.columns.tolist()
cols.insert(
    3, cols.pop(cols.index("new_word_span"))
)  # Move new_word_span to after the third column
cols.insert(
    4, cols.pop(cols.index("form_label"))
)  # Move form_label to after new_word_span
llm_samples_df = llm_samples_df[cols]
# Save back to JSON
llm_samples_df.to_json(
    OUTPUT_DIR / "genai_rewrites_sample.json",
    orient="records",
    indent=2,
    force_ascii=False,
)

In [11]:
display(llm_samples_df.head())

,id,candidates,chosen,rewritten,new_word_span,source_sentence,target_word,word_span,label
0,0,"[Euroopa, Ameerika, Rahvusvaheline, Ülemaailmne, Eesti, Põhja, Lõuna, Ühtne, Aasia, Kesk]",Euroopa,"Foto : Priit Simson Euroopa Komisjon tahab, et rohkem tossavad autod oleks kõrgemini maksustatud.","[20, 27]","Foto : Priit Simson Euroopa Komisjon tahab, et rohkem tossavad autod oleks kõrgemini maksustatud.",Euroopa,"[20, 27]",[sg g]
1,1,"[raske, keeruline, võimatu, tülikas, vaevaline, kurnav, koormav, vaevarikas, ebaselge, raskepärane]",raske,"Serbia-Montenegro laul oli väga lüüriline ja seal polnud mingit sõud, aga ometi tuli teiseks,"" on Naela sõnul raske öelda, milles täpselt võidulaulu võti peitub.","[110, 115]","Serbia-Montenegro laul oli väga lüüriline ja seal polnud mingit sõud, aga ometi tuli teiseks,"" on Naela sõnul raske öelda, milles täpselt võidulaulu võti peitub.",raske,"[110, 115]",[sg n]
2,2,"[majandusaasta, eelarveaasta, perioodi, kalendriaasta, tuleva, käesoleva, ülejärgmise, jooksva, tööaasta, uue]",majandusaasta,"Neljapäevase otsuse tulemusel jääb Tallinna selle majandusaasta eelarve suuruseks 2,26 miljardit krooni ehk ligikaudu sama palju kui enne juulis kinnitatud positiivset lisaeelarvet.","[50, 63]","Neljapäevase otsuse tulemusel jääb Tallinna selle aasta eelarve suuruseks 2,26 miljardit krooni ehk ligikaudu sama palju kui enne juulis kinnitatud positiivset lisaeelarvet.",aasta,"[50, 55]",[sg g]
3,3,"[Viljandi, Kehra, Tallinna, Aruküla, Tapa, Tartu, Pärnu, Viimsi, Raasiku, Valga]",Viljandi,Laupäeval kell 16.15 algab samas Eesti meeste meistrivõistluste kuues finaalkohtumine Reval-Sport/TTÜ ja Viljandi Serviti vahel.,"[105, 113]",Laupäeval kell 16.15 algab samas Eesti meeste meistrivõistluste kuues finaalkohtumine Reval-Sport/TTÜ ja Põlva Serviti vahel.,Põlva,"[105, 110]",[sg g]
4,4,"[keeruline, raske, võimatu, tülikas, vaevaline, ebaselge, küllaltki raske, probleemne, vaevarikas, kõva]",keeruline,"Nende põhjal on keeruline aru saada, mis tegelikult toimus.","[16, 25]","Nende põhjal on raske aru saada, mis tegelikult toimus.",raske,"[16, 21]",[sg n]


In [ ]:
# # Open LLM sample annotation data
# llm_samples_df = pd.read_json(
#     OUTPUT_DIR / "genai_rewrites_sample_100.json", orient="records"
# )

# new_records = []

# # For each record, iterate over the candidates, construct the candidate sentence, and perform morphological analysis to extract the case and other features of the candidate replacement word.
# for index, row in llm_samples_df.iterrows():
#     id = row["id"]
#     candidates = row.get("candidates", [])
#     chosen = row.get("chosen", "")
#     rewritten = row.get("rewritten", "")
#     new_word_span = row.get("new_word_span", None)
#     source_sentence = row["source_sentence"]
#     target_word = row["target_word"]
#     word_span = row["word_span"]
#     label = row.get("label", [])[
#         0
#     ]  # Assuming label is a list and we take the first element as the form label
#     # print(f"Source sentence: {source_sentence}")
#     # print(f"Target word span: {word_span}")
#     chosen_candidate = None
#     chosen_candidate_word_span = None
#     rewritten_sentence_w_candidate = None
#     for candidate in candidates:
#         # Construct sentence with the candidate replacement
#         candidate_sentence = (
#             source_sentence[: word_span[0]]
#             + candidate
#             + source_sentence[word_span[1] :]
#         )
#         # print(f"Candidate sentence: {candidate_sentence}")
#         # Create EstNLTK Text object and perform morphological analysis
#         estnltk_text = estnltk.Text(candidate_sentence)
#         estnltk_text.tag_layer("morph_analysis")
#         # Extract the morphological tags for the candidate word
#         morph_layer = estnltk_text.morph_analysis
#         # Find the token in the morph layer that corresponds to the candidate replacement
#         candidate_token = None
#         for token in morph_layer:
#             if (
#                 token.start <= word_span[0] < token.end
#             ):  # Check if the token covers the start of the original word span
#                 candidate_token = token
#                 break
#         if candidate_token:
#             # Check if the candidate is not the same as the original word
#             if candidate_token.text == target_word:
#                 continue
#             # Check if the candidate's form label matches with the given form label
#             candidate_label = candidate_token.form[0] if candidate_token.form else None
#             if candidate_label == label:
#                 chosen_candidate = candidate_token.text
#                 chosen_candidate_word_span = [
#                     candidate_token.start,
#                     candidate_token.end,
#                 ]
#                 rewritten_sentence_w_candidate = candidate_sentence
#                 break
#     if chosen_candidate is None:
#         print(f"No suitable candidate found for record ID: {id}")
#         continue
#     # Create a record for the chosen candidate and its features
#     candidate_record = {
#         "id": id,
#         "candidates": candidates,
#         "chosen": chosen_candidate,
#         "rewritten": rewritten_sentence_w_candidate,
#         "new_word_span": chosen_candidate_word_span,
#         "form_label": label,
#         "source_sentence": source_sentence,
#         "target_word": target_word,
#         "word_span": word_span,
#     }
#     new_records.append(candidate_record)

No suitable candidate found for record ID: 6
No suitable candidate found for record ID: 18
No suitable candidate found for record ID: 25
No suitable candidate found for record ID: 51
No suitable candidate found for record ID: 55


In [ ]:
# # Create a new DataFrame from the enriched records and save to JSON
# enriched_df = pd.DataFrame(new_records)
# enriched_df.to_json(
#     OUTPUT_DIR / "genai_rewrites_sample_100_enriched.json",
#     orient="records",
#     indent=2,
#     force_ascii=False,
# )

### Morphological annotation with LLM


In [5]:
# System prompt: enforce Estonian rewriting behaviour, case conditioning, and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.
Your task is to provide 10 context-appropriate alternatives to an exactly one marked token in angle brackets <...>.

Follow this procedure internally:
1. Infer the marked word’s grammatical role and morphological form from the sentence context.
2. Generate candidate replacements that are compatible with that inferred form.

Rules:
- Preserve tense, number, case, agreement, capitalisation, punctuation, and word order as much as possible.
- The replacement must fit the sentence context and the target word’s inferred morphology.
- The inferred morphological form of the replacement must be in one of these allowed cases only: nominative, genitive, partitive, or illative.
- Do not generate candidates that violate the required case.
- Do not repeat the original marked token as a candidate unless it is the only valid option.
- If the marked token is a proper name (person, place, organisation), replace it with another plausible proper name while still respecting the allowed-case constraint.
- Return strict JSON only. Do not add explanations or markdown.

JSON schema:
{
  "id": string,
  "candidates": array of 10 strings,
}
"""

# Few-shot example to anchor the output format
few_shot_user = """Sentence ID: 0
Sentence: Ma näen <maja>.
"""

few_shot_assistant = json.dumps(
    {
        "id": "0",
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "häärberit",
        ],
    },
    ensure_ascii=False,
    indent=2,
)

# Real input sentence
user_prompt = """Sentence ID: 1
Sentence: Samas on kõik uue kodu lähistel asuvad koolid sellised, mis võtavad <gümnaasiumi> vastu katsetega.
"""

contents = [
    types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
    types.Content(role="model", parts=[types.Part.from_text(text=few_shot_assistant)]),
    types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
]


# Create a user prompt template for the real sentences, using the same format as the few-shot example
def create_user_prompt(sentence_id, sentence, word_span):
    # Mark the target word with angle brackets
    start, end = word_span
    marked_sentence = (
        sentence[:start] + "<" + sentence[start:end] + ">" + sentence[end:]
    )
    return f"""Sentence ID: {sentence_id}
Sentence: {marked_sentence}
"""


def append_result_to_json(file_path, record, id=None):
    """Append one record to a JSON array stored in file_path."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    # If the file exists and is non-empty, load the existing data; otherwise start with an empty list
    if file_path.exists() and file_path.stat().st_size > 0:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = []
    else:
        data = []

    if not isinstance(data, list):
        data = []
    if id:
        # If ID is provided, check if a record with the same ID already exists and update it; otherwise append the new record
        existing_record = next((r for r in data if int(r.get("id")) == int(id)), None)
        if existing_record:
            existing_record.update(record)
            # Remove error field if it exists, since we have a successful rewrite now
            existing_record.pop("error", None)
        else:
            data.append(
                record
            )  # Append the new record if no existing record with the same ID is found
    else:
        # If no ID is provided, put the record at the end of the list
        data.append(record)
    # Write the updated list back to the file with pretty formatting
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def rewrite_sentence_with_genai(sentence_id, sentence, word_span, metadata):
    """Rewrite one sentence by replacing the marked target word via Gemini."""
    user_prompt = create_user_prompt(
        sentence_id=sentence_id, sentence=sentence, word_span=word_span
    )
    # The few-shot example is included in every call to ensure the model understands the expected JSON format and the task.
    local_contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
        types.Content(
            role="model", parts=[types.Part.from_text(text=few_shot_assistant)]
        ),
        types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
    ]
    # Request a JSON response from Gemini / Google Gen AI
    response = client.models.generate_content(
        model=model,
        contents=local_contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            # temperature=0.3,
            top_p=0.95,
            max_output_tokens=100000,
            response_mime_type="application/json",
        ),
    )
    # Parse the JSON response and enrich it with metadata for later analysis
    content = response.text or "{}"
    parsed = json.loads(content)

    # Keep useful metadata for later analysis/debugging.
    parsed["source_sentence"] = sentence
    parsed["target_word"] = metadata.get("target_word", "")
    if type(word_span) == list:
        parsed["word_span"] = word_span
    else:
        parsed["word_span"] = word_span.astype(
            int
        ).tolist()  # Convert numpy array to list for JSON serialization
    parsed["label"] = metadata.get("label", [])

    return parsed


def get_latest_processed_id(output_file):
    """Get the highest sentence ID that has already been processed and saved in the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        with open(output_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list) and len(data) > 0:
                return max(int(record["id"]) for record in data if "id" in record)
    return -1  # No valid records found, start from the beginning


def get_unprocessed_dataset(dataset, output_file):
    """Filter the dataset to include only records that have errors or have not been processed yet, based on the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        unprocessed_records = []
        processed_records = []
        # First, get the list of sentences that have errors in the output file
        with open(output_file, "r", encoding="utf-8") as f:
            processed_data = json.load(f)
            for record in processed_data:
                if "error" in record:
                    unprocessed_records.append(record["source_sentence"])
                else:
                    processed_records.append(record["source_sentence"])
        # Now filter the original dataset to include only records that are either unprocessed or have errors
        filtered_dataset = dataset[
            dataset["sentence"].isin(unprocessed_records)
            | ~dataset["sentence"].isin(processed_records)
        ]
        return filtered_dataset
    else:
        return dataset  # If no output file exists, return the entire dataset as unprocessed


def is_transient_error(exc: Exception) -> bool:
    """Heuristically detect retryable errors such as rate limit and network hiccups."""
    message = str(exc).lower()
    transient_markers = [
        "429",
        "rate",
        "quota",
        "too many requests",
        "timeout",
        "timed out",
        "connection",
        "temporar",
        "unavailable",
        "503",
        "502",
        "504",
        "internal",
    ]
    return any(marker in message for marker in transient_markers)


def rewrite_with_retry(sentence_id, sentence, word_span, metadata, config):
    """Call rewrite function with exponential backoff on transient errors."""
    for attempt in range(1, config["max_attempts"] + 1):
        try:
            return rewrite_sentence_with_genai(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                metadata=metadata,
            )
        except Exception as exc:
            # should_retry = is_transient_error(exc) and attempt < config["max_attempts"]
            should_retry = (
                attempt < config["max_attempts"]
            )  # Retry on all exceptions for robustness, but still limit the number of attempts
            if not should_retry:
                raise

            backoff = min(
                config["max_backoff_seconds"],
                config["initial_backoff_seconds"] * (2 ** (attempt - 1)),
            )
            sleep_s = backoff + random.uniform(0.0, config["jitter_seconds"])
            print(
                f"Retry {attempt}/{config['max_attempts'] - 1} for sentence_id={sentence_id} "
                f"after transient error: {exc}. Sleeping {sleep_s:.2f}s..."
            )
            time.sleep(sleep_s)

In [20]:
# Load the homonyms dataset to get the sentences we want to rewrite with Gen AI
homonyms_df = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_sentences.parquet"
)
display(homonyms_df.head())

,num,inflection_type,sentence,word,word_span,label,source
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [21]:
# Convert labels to strings
homonyms_df["label_str"] = homonyms_df["label"].apply(lambda x: x[0])

In [47]:
##### Create a small sample of sentences from homonym_df for testing the batch processing and error handling
max_rows = 200  # Change or remove this limit for larger runs

### Calculate the number of groups based on inflection type and form label
# First check the number of unique inflection types
num_inflection_types = homonyms_df["inflection_type"].nunique()
# Find the number of unique form labels for each inflection type
form_labels_per_inflection = homonyms_df.groupby("inflection_type")[
    "label_str"
].nunique()
print("Number of unique form labels per inflection type:")
print(form_labels_per_inflection)
print(
    f"Number of unique inflection types: {num_inflection_types}"
)  # 4 inflection types in total (1, 16, 17, 19)
print(
    f"Number of inflection types and form label groups: {form_labels_per_inflection.sum()}"  # 10 inflection type + form label groups in total
)
print()

### Extract n sentences from each inflection type (1, 16, 17, 19) and from each form label (sg n, sg g, sg p, adt) to have a balanced test set across both dimensions
cols = [c for c in homonyms_df.columns if c not in ["inflection_type", "label_str"]]
sample_df = (
    homonyms_df.groupby(["inflection_type", "label_str"])[cols]
    .apply(
        lambda group: group.sample(
            n=int(max_rows / form_labels_per_inflection.sum()),
            random_state=seed,
        )
    )
    .reset_index(level=[0, 1])  # bring group keys
    .reset_index()
)
# Display inflection type + form label distribution in the sample to verify balance
print("Inflection type and form label distribution in the sample:")
print(sample_df.groupby(["inflection_type", "label_str"]).size())
# Check few sample sentences and their metadata
display(sample_df.head())

Number of unique form labels per inflection type:
inflection_type
1     2
16    2
17    3
19    3
Name: label_str, dtype: int64
Number of unique inflection types: 4
Number of inflection types and form label groups: 10

Inflection type and form label distribution in the sample:
inflection_type  label_str
1                sg g         20
                 sg n         20
16               sg g         20
                 sg n         20
17               sg g         20
                 sg n         20
                 sg p         20
19               adt          20
                 sg g         20
                 sg p         20
dtype: int64


,index,inflection_type,label_str,num,sentence,word,word_span,label,source
0,878,1,sg g,1,"Chichester kandis esimese ehmatusega kaardile Austraalia lähema punkti Fremantlei, kuid tema lõi selja sirgeks.",Austraalia,"[46, 56]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,4623,1,sg g,2,"Väitekirja struktuur Tulenevalt väitekirja ülesehitusest õpik-monograafiana on töö eesmärkide saavutamiseks 1) uuritud maksukorralduse ja tarbimismaksude teoreetilisi aluseid ; 2) käsitletud käibemaksu rakendamise ajalugu ning põhjusi, mis mõjutasid selle rakendamist Euroopa Liidus või rakendamise edasilükkamist Ameerika Ühendriikides ; 3) analüüsitud käibemaksu puhul kasutatavat terminoloogiat ja käibemaksu rakendamise praktilisi probleeme ning selgitatud nende võimalikke lehendusi autori poolt väljatöötatud praktiliste näidetega ; 4) analüüsitud toiduainete käibemaksumäära alandamise mõju leibkondade elujärjele võrrelduna tulumaksuvaba miinimumi tõstmisega kaasneva mõjuga.",Ameerika,"[314, 322]",[sg g],infl_type_01_1000_v2_project-5-at-2024-12-11-21-53-280753b4.json
2,387,1,sg g,1,"Piritalt Taevaskotta Oleme kolmekümne aasta jooksul ""Reliikviat"" korduvalt näinud, ent tema menu saladuse mõistmiseks sellest ei piisa.",aasta,"[38, 43]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,884,1,sg g,1,Kanada rajasid inglastest ja prantslastest väljarändajad.,Kanada,"[0, 6]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,4805,1,sg g,2,"Janek Luts hea tämbri ja rahuliku ning häirimatu hääletaustaga sobib raadiosse suurepäraselt ja ehk isegi kõige enam ka seetõttu, et ei oma aukartustäratavat aurat.",häirimatu,"[39, 48]",[sg g],infl_type_01_1000_v2_project-5-at-2024-12-11-21-53-280753b4.json


In [69]:
output_file = (
    OUTPUT_DIR / f"genai_rewrites_sample_v3_{max_rows}.json"
)  # Output file for results

# gemini-3.1-flash-lite-preview has strict request limits, so we throttle and retry safely.
config = {
    # Pacing configuration to avoid hitting rate limits
    "base_delay_seconds": round(
        60 / 15, 2
    ),  # RPM is 15 for gemini-3.1-flash-lite-preview
    "jitter_seconds": 1.0,
    # Retry configuration for transient failures
    "max_attempts": 6,
    "initial_backoff_seconds": 2.0,
    "max_backoff_seconds": 60.0,
}

# Start from the next unprocessed sentence based on the output file contents

# latest_id = get_latest_processed_id(output_file)
# unprocessed_df = sample_df[sample_df.index > latest_id]
# print(
#     f"Starting processing from sentence ID > {latest_id + 1}. Total unprocessed sentences: {len(unprocessed_df)}"
# )
unprocessed_df = get_unprocessed_dataset(sample_df, output_file)
if len(unprocessed_df) == 0:
    print("No unprocessed sentences found. All done!")
else:
    print(f"Total unprocessed sentences to process: {len(unprocessed_df)}")
    print("Next sentence to process:")
    display(unprocessed_df.head(1))

Total unprocessed sentences to process: 191
Next sentence to process:


,index,inflection_type,label_str,num,sentence,word,word_span,label,source
0,878,1,sg g,1,"Chichester kandis esimese ehmatusega kaardile Austraalia lähema punkti Fremantlei, kuid tema lõi selja sirgeks.",Austraalia,"[46, 56]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [67]:
# Find processed records that are in the sample_df
if output_file.exists() and output_file.stat().st_size > 0:
    with open(output_file, "r", encoding="utf-8") as f:
        processed_data = json.load(f)
        processed_ids = [
            int(record["id"]) for record in processed_data if "id" in record
        ]
        processed_sentences = sample_df[sample_df["index"].isin(processed_ids)]
        print(
            f"Number of sentences in sample_df that have already been processed: {len(processed_sentences)}"
        )
        print("Sample of processed sentences:")
        display(sample_df[sample_df["index"].isin(processed_ids)].head(10))
else:
    print("No output file found, so no sentences have been processed yet.")

# Now remove all the already processed sentences from the output file that are not in the sample_df
if output_file.exists() and output_file.stat().st_size > 0:
    with open(output_file, "r", encoding="utf-8") as f:
        processed_data = json.load(f)
        processed_ids = [
            int(record["id"]) for record in processed_data if "id" in record
        ]
        filtered_data = [
            record
            for record in processed_data
            if int(record.get("id", -1)) in sample_df["index"].values
        ]
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(filtered_data, f, ensure_ascii=False, indent=2)

Number of sentences in sample_df that have already been processed: 9
Sample of processed sentences:


,index,inflection_type,label_str,num,sentence,word,word_span,label,source
11,4502,1,sg g,2,"Teisipäeval, 10. veebruaril teatas Indoneesia esindajatekoja asespiiker Abdul Gafur, et riigi parlament toetab valitsuse ettepanekut rakendada Indoneesia valuuta, ruupia stabiliseerimiseks valuutakomitee põhimõtet.",ruupia,"[163, 169]",[sg g],infl_type_01_1000_v2_project-5-at-2024-12-11-21-53-280753b4.json
33,630,1,sg n,1,"Aga samas on ju täiesti selge, et kool peab olema koht, kus turvatunne oleks tagatud kõigile, nii õpilastele kui õpetajatele.",selge,"[24, 29]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
50,5868,16,sg g,2,"Uue auto mootorivalikusse kuuluvad seitse mootorit, esialgu on valik väiksem.",auto,"[4, 8]",[sg g],infl_type_16_1000_v2_project-5-at-2025-01-03-01-42-87bca577.json
58,5587,16,sg g,2,"Kell 21. 30 leidis politseipatrull Vilde teelt trepikojast 36-aastase Vitali surnukeha, keda oli löödud noaga südamesse.",Vitali,"[70, 76]",[sg g],infl_type_16_1000_v2_project-5-at-2025-01-03-01-42-87bca577.json
87,6732,17,sg g,2,Seni eksportis firma suurema osa oma toodangust Venemaale.,osa,"[29, 32]",[sg g],infl_type_17_1000_v2.json
123,6741,17,sg p,2,"""Tahame, et sellest saaksid osa võimalikult paljud väiksed spordi- ja loomasõbrad,"" ütles Rüütmann.",osa,"[28, 31]",[sg p],infl_type_17_1000_v2.json
132,6544,17,sg p,2,"Preatoni ja teiste välisinvestorite seisukohast vaadatuna ei ole turismiäri ajamine Eestis tõesti eriti atraktiivne, sest maailmakodanikena saavad nad paigutada oma raha just sinna maailmanurka ja tegevusalasse, kus riski ja tulu vahekord on parajasti kõige soodsam.",raha,"[165, 169]",[sg p],infl_type_17_1000_v2.json
169,3289,19,sg g,1,"Huvitavatel harmooniatel liikuva heliloomingu vormistab meeldejäävaks tervikuks noore lauljatari Kene Verniku vokaal, mida on võrreldud nii Björki, Sandra Nasicu kui ka Siouxsie Siouxsiga.",lauljatari,"[86, 96]",[sg g],infl_type_19_1000_v1_project-6-at-2025-11-15-14-13-42753676.json
193,7388,19,sg p,2,Ema koolitab üksi kolme last Arno Tali sihtkapitali tegevdirektori Loora Tarmi sõnul soovis tänavu stipendiumi saada üle neljasaja noore.,stipendiumi,"[99, 110]",[sg p],infl_type_19_1000_v2_project-7-at-2025-11-21-23-02-24f590e4.json


In [70]:
# Batch rewrite loop: call Gen AI and append each result to a JSON file
processed = 0
for i, (sentence_id, sentence, word_span, word, label) in enumerate(
    zip(
        unprocessed_df["index"],
        unprocessed_df["sentence"],
        unprocessed_df["word_span"],
        unprocessed_df["word"],
        unprocessed_df["label"],
    )
):
    try:
        metadata = {
            "target_word": word,
            "label": label.tolist(),  # Convert numpy array to list for JSON serialization
        }
        result = rewrite_with_retry(
            sentence_id=sentence_id,
            sentence=sentence,
            word_span=word_span,
            config=config,
            metadata=metadata,
        )
        append_result_to_json(output_file, result, id=sentence_id)
        print(f"[{processed + 1}] Saved sentence_id={sentence_id}")
    except Exception as exc:
        error_record = {
            "id": str(sentence_id),
            "source_sentence": sentence,
            "target_word": word,
            "word_span": word_span.astype(int).tolist(),
            "label": label.tolist(),
            "error": str(exc),
        }
        append_result_to_json(output_file, error_record, id=sentence_id)
        print(f"[{processed + 1}] Error on sentence_id={sentence_id}: {exc}")

    processed += 1

    # Additional pacing between successful/failed items to avoid bursty traffic.
    if i < max_rows:
        sleep_s = config["base_delay_seconds"] + random.uniform(
            0.0, config["jitter_seconds"]
        )
        print(f"Sleeping {sleep_s:.2f}s before next request...")
        time.sleep(sleep_s)

print(f"Finished. Processed {processed} rows. Results appended to: {output_file}")

INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[1] Saved sentence_id=878
Sleeping 4.38s before next request...
INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[2] Saved sentence_id=4623
Sleeping 4.75s before next request...
INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
[3] Saved sentence_id=387
Sleeping 4.35s before next request...
INFO:models.py:6013: AFC is enabled with max remote calls: 10.
INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.goog

In [ ]:
# Batch rewrite loop for error records: retry only the sentences that previously failed, with the same pacing and retry logic.
if output_file.exists():
    with open(output_file, "r", encoding="utf-8") as f:
        data = json.load(f)
        error_records = [record for record in data if "error" in record]
        print(f"Found {len(error_records)} error records to retry.")
        for record in tqdm.tqdm(error_records, desc="Retrying error records"):
            sentence_id = int(record["id"])
            sentence = record["source_sentence"]
            word_span = record["word_span"]
            target_word = record["target_word"]
            label = record.get("label", [])
            metadata = {
                "target_word": target_word,
                "label": label,
            }
            try:
                result = rewrite_with_retry(
                    sentence_id=sentence_id,
                    sentence=sentence,
                    word_span=word_span,
                    config=config,
                    metadata=metadata,
                )
                # Update the original record with the new result and remove the error field
                record.update(result)
                record.pop("error", None)
                print(f"Successfully retried sentence_id={sentence_id}")
            except Exception as exc:
                record["error"] = str(exc)
                print(f"Retry failed for sentence_id={sentence_id}: {exc}")
            # Save after each retry attempt to ensure progress is not lost
            append_result_to_json(output_file, record)

Found 4 error records to retry.


Retrying error records:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:models.py:6013: AFC is enabled with max remote calls: 10.


Retrying error records:  25%|██▌       | 1/4 [00:01<00:03,  1.02s/it]

INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
Successfully retried sentence_id=73
INFO:models.py:6013: AFC is enabled with max remote calls: 10.


...

INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
Successfully retried sentence_id=91
INFO:models.py:6013: AFC is enabled with max remote calls: 10.


Retrying error records:  75%|███████▌  | 3/4 [00:02<00:00,  1.09it/s]

INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
Retry failed for sentence_id=73: Expecting ',' delimiter: line 8 column 15 (char 113)
INFO:models.py:6013: AFC is enabled with max remote calls: 10.


Retrying error records: 100%|██████████| 4/4 [00:03<00:00,  1.11it/s]

INFO:_client.py:1025: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
Successfully retried sentence_id=91


In [71]:
# Open LLM sample annotation data
llm_samples_df = pd.read_json(output_file, orient="records")

new_records = []

# For each record, iterate over the candidates, construct the candidate sentence, and perform morphological analysis to extract the case and other features of the candidate replacement word. Assume each candidate has uniform probability and sum up the probabilities for candidates that match each form label to get a distribution of form labels for the candidates.
for index, row in tqdm.tqdm(llm_samples_df.iterrows(), total=llm_samples_df.shape[0]):
    id = row["id"]
    candidates = row.get("candidates", [])
    chosen = row.get("chosen", "")
    rewritten = row.get("rewritten", "")
    new_word_span = row.get("new_word_span", None)
    source_sentence = row["source_sentence"]
    target_word = row["target_word"]
    word_span = row["word_span"]
    label = row.get("label", [])[
        0
    ]  # Assuming label is a list and we take the first element as the form label
    # print(f"Source sentence: {source_sentence}")
    # print(f"Target word span: {word_span}")
    candidate_form_weight = (
        1 / len(candidates) if candidates else 0
    )  # Uniform weight for each candidate
    form_probabilities = {}
    for candidate in candidates:
        # Construct sentence with the candidate replacement
        candidate_sentence = (
            source_sentence[: word_span[0]]
            + candidate
            + source_sentence[word_span[1] :]
        )
        # print(f"Candidate sentence: {candidate_sentence}")
        # Create EstNLTK Text object and perform morphological analysis
        estnltk_text = estnltk.Text(candidate_sentence)
        estnltk_text.tag_layer("morph_analysis")
        # Extract the morphological tags for the candidate word
        morph_layer = estnltk_text.morph_analysis
        # Find the token in the morph layer that corresponds to the candidate replacement
        candidate_token = None
        for token in morph_layer:
            if (
                token.start <= word_span[0] < token.end
            ):  # Check if the token covers the start of the original word span
                candidate_token = token
                break
        if candidate_token:
            # Count up the form labels for the candidate
            number_of_labels = len(candidate_token.form) if candidate_token.form else 0
            if number_of_labels > 0:
                weight_per_label = candidate_form_weight / number_of_labels
                for candidate_label in candidate_token.form:
                    form_probabilities[candidate_label] = (
                        form_probabilities.get(candidate_label, 0) + weight_per_label
                    )
            else:
                form_probabilities["unknown"] = (
                    form_probabilities.get("unknown", 0) + candidate_form_weight
                )

    best_form_label = (
        max(form_probabilities, key=form_probabilities.get)
        if form_probabilities
        else "unknown"
    )
    # Create a record for the chosen candidate and its features
    candidate_record = {
        "id": id,
        "candidates": candidates,
        "pred_label": best_form_label,
        "true_label": label,
        "predicted_form_distribution": form_probabilities,
        "source_sentence": source_sentence,
        "target_word": target_word,
        "word_span": word_span,
    }
    new_records.append(candidate_record)

  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [00:13<00:00, 14.90it/s]


In [72]:
# Create a new DataFrame from the predicted form labels and their distributions and save to JSON
homonym_results_df = pd.DataFrame(new_records)
homonym_results_df_path = (
    HOMONYMS_DIRS["processed"] / f"homonyms_llm_mlm_predictions_v3_{max_rows}.parquet"
)
homonym_results_df.to_parquet(homonym_results_df_path, index=False)